In [0]:
LOGIN = "adam151212"
CATALOG = "dbr_dev"
SCHEMA = LOGIN
API_BASE = "http://3.78.187.66:8000"
SCHEMA_PATH = f"{CATALOG}.{SCHEMA}"

In [0]:

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_PATH}")
display(spark.sql(f"SHOW SCHEMAS IN {CATALOG}"))

In [0]:
import pandas as pd

pdf = pd.read_csv("../camera_dim.csv")
camera_dim = spark.createDataFrame(pdf)

(camera_dim.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{SCHEMA_PATH}.camera_dim"))
print(f"Rows: ", spark.table(f"{SCHEMA_PATH}.camera_dim").count())
display(spark.table(f"{SCHEMA_PATH}.CAMERA_DIM").limit(5))

In [0]:
import json, requests

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType

# The API provides unrestricted output (recordings are randomly sampled. the 3 source datasets contain approximately 1.06M rows in total).
# ~50k events is the minimum for a representative dashboard, thus 60k gives a safe margin
TARGET_EVENTS = 60000
BATCH = 20000

requests.post(f"{API_BASE}/api/v1/reset", timeout=30)

rows = []
while len(rows) < TARGET_EVENTS:
    body = requests.get(f"{API_BASE}/api/v1/telemetry",
                        params={"limit": BATCH}, timeout=60).json()
    for e in body["events"]:
        p = e.get("payload", {})
        rows.append((
            e["event_type"],
            e["device_id"],
            int(e["timestamp"]),
            p.get("camera"),
            json.dumps(p)
        ))
    if body.get("done"):
        break
print("fetched events: ", len(rows))

schema = StructType([
    StructField("event_type", StringType()),
    StructField("device_id", StringType()),
    StructField("ts_unix", LongType()),
    StructField("camera", StringType()),
    StructField("payload", StringType()),
])

telemetry = (spark.createDataFrame(rows, schema)
             .withColumn("event_time", F.timestamp_seconds("ts_unix"))
             .withColumn("event_date", F.to_date("event_time")))

(telemetry.write.format("delta").mode("overwrite")
             .option("overwriteSchema", "true")
             .saveAsTable(f"{SCHEMA_PATH}.telemetry_raw"))

print("telemetry_raw rows: ", spark.table(f"{SCHEMA_PATH}.telemetry_raw").count())
display(spark.table(f"{SCHEMA_PATH}.telemetry_raw").limit(5))

In [0]:
# SELECT
detections = (spark.table(f"{SCHEMA_PATH}.telemetry_raw")
    .filter(F.col("event_type") == "AI_DETECTION_BOX")
    .select(
        "device_id",
        "camera",
        F.get_json_object("payload", "$.object_type").alias("object_type"),
        F.get_json_object("payload", "$.box_coordinates").alias("box"),
        F.get_json_object("payload", "$.confidence").cast("double").alias("confidence"),
        "event_time",
    ))
display(detections)


In [0]:
#GROUP  BY
display(telemetry.groupBy("event_date", "event_type").count()
        .orderBy("event_date", "event_type"))


In [0]:

# FILTER
detections = (telemetry
    .filter(F.col("event_type") == "AI_DETECTION_BOX")
    .withColumn("object_type", F.get_json_object("payload", "$.object_type"))
    .withColumn("confidence",  F.get_json_object("payload", "$.confidence").cast("double"))
    .filter(F.col("confidence") >= 0.8))
display(detections.select("camera", "object_type", "confidence", "event_time"))


In [0]:
#JOIN
camera_dim = spark.table(f"{SCHEMA_PATH}.camera_dim").drop("device_id")
enriched = telemetry.join(camera_dim, on="camera", how="inner")

results = (enriched
    .filter(F.col("event_type") == "AI_DETECTION_BOX")
    .groupBy("event_date", "location", "zone", "quality", "device_id")
    .agg(F.count("*").alias("detections"))
    .orderBy("event_date", "location"))

(results.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SCHEMA_PATH}.results"))

print("rows: ", spark.table(f"{SCHEMA_PATH}.results").count())
display(spark.table(f"{SCHEMA_PATH}.results"))

**Dashboard**


In [0]:
metrics = (telemetry
    .filter(F.col("event_type") == "SYSTEM_METRICS")
    .withColumn("cpu",       F.get_json_object("payload", "$.cpu_usage_pct").cast("double"))
    .withColumn("ram",       F.get_json_object("payload", "$.ram_usage_pct").cast("double"))
    .withColumn("disk_temp", F.get_json_object("payload", "$.disk_temperature_c").cast("double"))
    .withColumn("disk_used", F.get_json_object("payload", "$.disk_used_pct").cast("double")))

metrics_summary = (metrics.groupBy("device_id").agg(
    F.round(F.avg("cpu"),  1).alias("avg_cpu_pct"),
    F.round(F.avg("ram"),  1).alias("avg_ram_pct"),
    F.round(F.avg("disk_temp"), 1).alias("avg_disk_temp_c"),
    F.round(F.avg("disk_used"), 1).alias("avg_disk_used_pct"),
    F.count("*").alias("samples"),
))

(metrics_summary.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SCHEMA_PATH}.metrics_summary"))

display(metrics_summary)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
display(spark.sql(f"""
    SELECT cd.device_id,
           ROUND(SUM(CASE WHEN cd.quality='HD'
                    THEN CAST(get_json_object(t.payload,'$.segment_size_mb') AS DOUBLE) END)/1024, 2) AS hd_gb,
           ROUND(SUM(CASE WHEN cd.quality='SD'
                    THEN CAST(get_json_object(t.payload,'$.segment_size_mb') AS DOUBLE) END)/1024, 2) AS sd_gb
    FROM {SCHEMA_PATH}.telemetry_raw t
    JOIN {SCHEMA_PATH}.camera_dim cd ON t.camera = cd.camera
    WHERE t.event_type = 'RECORDINGS_METADATA'
    GROUP BY cd.device_id
"""))

Databricks visualization. Run in Databricks to view.

In [0]:
display(spark.sql(f"""
    SELECT get_json_object(payload,'$.object_type') AS object_type, COUNT(*) AS detections
    FROM {SCHEMA_PATH}.telemetry_raw
    WHERE event_type='AI_DETECTION_BOX'
    GROUP BY object_type ORDER BY detections DESC
"""))

Databricks visualization. Run in Databricks to view.

In [0]:
display(spark.sql(f"""
    SELECT cd.location, COUNT(*) AS detections
    FROM {SCHEMA_PATH}.telemetry_raw t
    JOIN {SCHEMA_PATH}.camera_dim cd ON t.camera = cd.camera
    WHERE t.event_type='AI_DETECTION_BOX'
    GROUP BY cd.location ORDER BY detections DESC
"""))

Databricks visualization. Run in Databricks to view.

In [0]:
display(spark.sql(f"""
    SELECT event_date, device_id,
           ROUND(AVG(CAST(get_json_object(payload,'$.cpu_usage_pct') AS DOUBLE)),1) AS avg_cpu
    FROM {SCHEMA_PATH}.telemetry_raw
    WHERE event_type='SYSTEM_METRICS' AND event_date >= '2026-05-11'
    GROUP BY event_date, device_id ORDER BY event_date
"""))

Databricks visualization. Run in Databricks to view.

In [0]:
display(spark.sql(f"""
    SELECT zone, SUM(detections) AS total
    FROM {SCHEMA_PATH}.results GROUP BY zone
"""))

Databricks visualization. Run in Databricks to view.